# l-v products (b=0 strip and densification)

Split out from `scan_load.ipynb` (Sections 8-9). Run `scan_load.ipynb`
through the QA cell first, then run this notebook in the same kernel
(or load the pickled `cell_combined`, `qa_flagged_set`, `v_lsr_overlap`,
`dv_kms` if running standalone).

## 8. Longitude-velocity diagram (b ~ 0)

Stack T_B(v) for cells near the galactic plane, average within longitude
bins, and display as a heatmap of v_LSR vs galactic longitude.

In [ ]:
import matplotlib as mpl

LV_DL = 2.0             # longitude bin width (deg)
LV_VMIN, LV_VMAX = -150, 130   # velocity display range (km/s, LSR)

# --- Inaccessible-longitude mask (Leuschner alt/az limits) ---
LEO_LAT_RAD = np.deg2rad(37.9183)
MIN_ALT_RAD = np.deg2rad(17.0)
MAX_ALT_RAD = np.deg2rad(83.0)
AZ_MIN_RAD = np.deg2rad(7.0)
AZ_MAX_RAD = np.deg2rad(348.0)
HA_STEPS = np.deg2rad(np.arange(0, 360, 1))


def never_observable_mask(l_arr, b_val):
    """True at each l in l_arr where (l, b_val) is never observable."""
    gc = ac.SkyCoord(l=l_arr * u_ast.deg,
                     b=np.full_like(l_arr, b_val, dtype=float) * u_ast.deg,
                     frame='galactic')
    dec_rad = np.deg2rad(gc.transform_to(ac.ICRS()).dec.deg)
    observable = np.zeros(len(l_arr), dtype=bool)
    for ha in HA_STEPS:
        sin_alt = (np.sin(LEO_LAT_RAD) * np.sin(dec_rad)
                   + np.cos(LEO_LAT_RAD) * np.cos(dec_rad) * np.cos(ha))
        alt = np.arcsin(np.clip(sin_alt, -1, 1))
        cos_az_num = np.sin(dec_rad) - np.sin(LEO_LAT_RAD) * sin_alt
        cos_az_den = np.cos(LEO_LAT_RAD) * np.cos(alt)
        with np.errstate(invalid='ignore', divide='ignore'):
            cos_az = np.clip(cos_az_num / cos_az_den, -1, 1)
        az = np.arccos(cos_az)
        az = np.where(np.sin(ha) > 0, 2 * np.pi - az, az)
        observable |= ((alt >= MIN_ALT_RAD) & (alt <= MAX_ALT_RAD)
                       & (az >= AZ_MIN_RAD) & (az <= AZ_MAX_RAD))
    return ~observable


def _bin_T_B(keys, lo_edges):
    """Return (image, n_per_bin) for the given cell keys.

    `lo_edges` are the longitude-bin edges in the WRAPPED frame [-180, 180).
    """
    nv_local = len(v_lsr_overlap)
    centers = 0.5 * (lo_edges[:-1] + lo_edges[1:])
    img = np.full((nv_local, len(centers)), np.nan)
    n_per = np.zeros(len(centers), dtype=int)
    if not keys:
        return img, n_per
    gl_w = np.array([((k[0] + 180.0) % 360.0) - 180.0 for k in keys])
    for i in range(len(centers)):
        in_bin = [k for k, gl in zip(keys, gl_w)
                  if lo_edges[i] <= gl < lo_edges[i + 1]]
        if not in_bin:
            continue
        stack = np.array([cell_combined[k]['T_B'] for k in in_bin])
        col_has = np.any(np.isfinite(stack), axis=0)
        if col_has.any():
            img[col_has, i] = np.nanmean(stack[:, col_has], axis=0)
        n_per[i] = len(in_bin)
    return img, n_per


# --- Pick cells at b=0, b=-1, b=+1 (excluding QA-flagged) ---
keys_b0 = [k for k in cell_combined if k[1] == 0  and k not in qa_flagged_set]
keys_bm = [k for k in cell_combined if k[1] == -1 and k not in qa_flagged_set]
keys_bp = [k for k in cell_combined if k[1] == 1  and k not in qa_flagged_set]

# Common longitude grid spans the union of all three latitudes
all_gl = np.array([k[0] for k in keys_b0 + keys_bm + keys_bp])
all_gl_w = ((all_gl + 180.0) % 360.0) - 180.0
l_lo = np.floor(all_gl_w.min() / LV_DL) * LV_DL
l_hi = np.ceil(all_gl_w.max() / LV_DL) * LV_DL
l_edges = np.arange(l_lo, l_hi + LV_DL, LV_DL)
l_centers = 0.5 * (l_edges[:-1] + l_edges[1:])

img_b0, n_b0 = _bin_T_B(keys_b0, l_edges)
img_bm, n_bm = _bin_T_B(keys_bm, l_edges)
img_bp, n_bp = _bin_T_B(keys_bp, l_edges)

# Interpolation pass: fill columns where b=0 has no coverage but BOTH
# b=-1 and b=+1 do. Average the two flanks.
interp_mask = (n_b0 == 0) & (n_bm > 0) & (n_bp > 0)
lv_image = img_b0.copy()
n_total = n_b0.copy()
for i in np.where(interp_mask)[0]:
    avg = 0.5 * (img_bm[:, i] + img_bp[:, i])
    valid = np.isfinite(avg)
    lv_image[valid, i] = avg[valid]
    n_total[i] = n_bm[i] + n_bp[i]

# Trim leading/trailing empty longitude bins
nonempty = n_total > 0
if nonempty.any():
    j0 = int(np.argmax(nonempty))
    j1 = len(nonempty) - int(np.argmax(nonempty[::-1]))
    lv_image = lv_image[:, j0:j1]
    interp_trim = interp_mask[j0:j1]
    l_centers_trim = l_centers[j0:j1]
    l_edges_trim = l_edges[j0:j1 + 1]
else:
    interp_trim = interp_mask
    l_centers_trim = l_centers
    l_edges_trim = l_edges

# Velocity bin edges
v_step = v_lsr_overlap[1] - v_lsr_overlap[0]
v_edges = np.concatenate([
    [v_lsr_overlap[0] - 0.5 * v_step],
    0.5 * (v_lsr_overlap[:-1] + v_lsr_overlap[1:]),
    [v_lsr_overlap[-1] + 0.5 * v_step],
])

finite_vals = lv_image[np.isfinite(lv_image)]
vmax = np.nanpercentile(finite_vals, 99) if finite_vals.size else 1.0
vmin = max(0.0, np.nanpercentile(finite_vals, 1)) if finite_vals.size else 0.0

fig, ax = plt.subplots(figsize=(11, 6))
pcm = ax.pcolormesh(l_edges_trim, v_edges, lv_image,
                    cmap='inferno', vmin=vmin, vmax=vmax, shading='flat')

# Mark interpolated columns with a faint white cross-hatch
v_lo_lim, v_hi_lim = LV_VMIN, LV_VMAX
first_interp = True
for i, is_interp in enumerate(interp_trim):
    if not is_interp:
        continue
    label = 'b=0 interpolated from b=+/-1' if first_interp else None
    with mpl.rc_context({'hatch.color': 'white', 'hatch.linewidth': 0.4}):
        ax.fill_between([l_edges_trim[i], l_edges_trim[i + 1]],
                        v_lo_lim, v_hi_lim,
                        facecolor='none', edgecolor='none',
                        hatch='xxx', alpha=0.5, zorder=3, label=label)
    first_interp = False

# Shade longitudes that are never observable from Leuschner at b = 0.
inacc = never_observable_mask(l_centers_trim.astype(float) % 360, 0.0)
runs = []
k = 0
while k < len(l_centers_trim):
    if inacc[k]:
        k0 = k
        while k < len(l_centers_trim) and inacc[k]:
            k += 1
        runs.append((l_edges_trim[k0], l_edges_trim[k]))
    else:
        k += 1
first_patch = True
for l0, l1 in runs:
    label = 'Never observable' if first_patch else None
    ax.fill_between([l0, l1], v_lo_lim, v_hi_lim,
                    color='red', alpha=0.08, zorder=4)
    with mpl.rc_context({'hatch.color': 'red', 'hatch.linewidth': 0.5}):
        ax.fill_between([l0, l1], v_lo_lim, v_hi_lim,
                        facecolor='none', edgecolor='none',
                        hatch='///', alpha=0.4, zorder=5, label=label)
    first_patch = False

ax.set_xlabel('Galactic longitude l [deg]')
ax.set_ylabel(r'$v_{\rm LSR}$ [km s$^{-1}$]')
ax.set_ylim(LV_VMIN, LV_VMAX)
ax.invert_xaxis()  # convention: l increases to the left
n_meas_bins = int((n_b0 > 0).sum())
n_interp_bins = int(interp_mask.sum())
ax.set_title(f'l-v diagram at b = 0   '
             f'({n_meas_bins} measured + {n_interp_bins} interpolated bins, '
             f'{LV_DL} deg)')
if runs or n_interp_bins > 0:
    ax.legend(loc='upper right', fontsize=9, framealpha=0.9)
cbar = fig.colorbar(pcm, ax=ax, pad=0.02)
cbar.set_label(r'$T_B$ [K]')
plt.tight_layout()
plt.show()

## 9. Densify the b=0 strip

The l-v image from Section 8 has gaps between sampled longitudes (some
real, some inside the inaccessible region). Linearly interpolate T_B
along longitude at each velocity channel to produce a densified b=0
strip. Constraints:

- Resample onto a finer longitude grid (`LV_DL_FINE = 0.5 deg`).
- Only fill gaps **inside contiguous observable segments** -- do not
  bridge across the never-observable wedge or extrapolate past the
  edges of the survey.
- Operate per velocity channel independently.

In [ ]:
LV_DL_FINE = 0.5   # densified longitude bin width (deg)

# Mask of observable longitudes on the SOURCE (coarse) grid.
inacc_src = never_observable_mask(l_centers_trim.astype(float) % 360, 0.0)

# Identify contiguous observable segments on the coarse grid. Within
# each segment we have at least the columns flagged as populated in
# Section 8 (n_total > 0 there); we'll interpolate strictly inside
# each segment.
n_total_trim = n_total[j0:j1] if nonempty.any() else n_total
populated = (n_total_trim > 0) & (~inacc_src)

segments = []
k = 0
while k < len(l_centers_trim):
    if populated[k] and not inacc_src[k]:
        k0 = k
        while (k < len(l_centers_trim)
               and not inacc_src[k]):
            k += 1
        # The segment spans [k0, k); find the actual data extent.
        seg_idx = np.arange(k0, k)
        seg_pop = seg_idx[populated[seg_idx]]
        if seg_pop.size >= 2:
            segments.append((seg_pop[0], seg_pop[-1] + 1))
    else:
        k += 1

# Build a finer longitude grid covering the same wrapped range.
l_lo_fine = l_centers_trim[0]
l_hi_fine = l_centers_trim[-1]
l_fine = np.arange(l_lo_fine, l_hi_fine + LV_DL_FINE, LV_DL_FINE)
nv = len(v_lsr_overlap)
lv_image_dense = np.full((nv, len(l_fine)), np.nan)

for seg_start, seg_stop in segments:
    l_seg = l_centers_trim[seg_start:seg_stop]
    img_seg = lv_image[:, seg_start:seg_stop]
    # Bins of l_fine inside this segment's coarse-grid extent.
    in_seg = (l_fine >= l_seg[0]) & (l_fine <= l_seg[-1])
    if not in_seg.any():
        continue
    l_target = l_fine[in_seg]
    for j in range(nv):
        row = img_seg[j]
        good = np.isfinite(row)
        if good.sum() < 2:
            continue
        lv_image_dense[j, in_seg] = np.interp(
            l_target, l_seg[good], row[good],
            left=np.nan, right=np.nan,
        )

n_filled_dense = int(np.isfinite(lv_image_dense).any(axis=0).sum())
print(f'Densified strip: {len(l_fine)} l-bins at {LV_DL_FINE} deg, '
      f'{n_filled_dense} populated')

# --- Visualize the densified strip ---
v_step_d = v_lsr_overlap[1] - v_lsr_overlap[0]
v_edges_d = np.concatenate([
    [v_lsr_overlap[0] - 0.5 * v_step_d],
    0.5 * (v_lsr_overlap[:-1] + v_lsr_overlap[1:]),
    [v_lsr_overlap[-1] + 0.5 * v_step_d],
])
l_edges_fine = np.concatenate([
    [l_fine[0] - 0.5 * LV_DL_FINE],
    0.5 * (l_fine[:-1] + l_fine[1:]),
    [l_fine[-1] + 0.5 * LV_DL_FINE],
])

finite_dense = lv_image_dense[np.isfinite(lv_image_dense)]
vmax_d = np.nanpercentile(finite_dense, 99) if finite_dense.size else 1.0
vmin_d = max(0.0, np.nanpercentile(finite_dense, 1)) if finite_dense.size else 0.0

fig, ax = plt.subplots(figsize=(11, 6))
pcm = ax.pcolormesh(l_edges_fine, v_edges_d, lv_image_dense,
                    cmap='inferno', vmin=vmin_d, vmax=vmax_d, shading='flat')
inacc_fine = never_observable_mask(l_fine.astype(float) % 360, 0.0)
runs = []
k = 0
while k < len(l_fine):
    if inacc_fine[k]:
        k0 = k
        while k < len(l_fine) and inacc_fine[k]:
            k += 1
        runs.append((l_edges_fine[k0], l_edges_fine[k]))
    else:
        k += 1
for l0, l1 in runs:
    ax.fill_between([l0, l1], LV_VMIN, LV_VMAX,
                    color='red', alpha=0.08, zorder=3)
    with mpl.rc_context({'hatch.color': 'red', 'hatch.linewidth': 0.5}):
        ax.fill_between([l0, l1], LV_VMIN, LV_VMAX,
                        facecolor='none', edgecolor='none',
                        hatch='///', alpha=0.4, zorder=4)
ax.set_xlabel('Galactic longitude l [deg]')
ax.set_ylabel(r'$v_{\rm LSR}$ [km s$^{-1}$]')
ax.set_ylim(LV_VMIN, LV_VMAX)
ax.invert_xaxis()
ax.set_title(f'Densified b=0 strip  ({LV_DL_FINE} deg longitude bins, '
             f'linear interp per channel)')
cbar = fig.colorbar(pcm, ax=ax, pad=0.02)
cbar.set_label(r'$T_B$ [K]')
plt.tight_layout()
plt.show()

## 10. Top-down view of the Milky Way

Kinematic deprojection of the densified b=0 strip with a flat rotation
curve (V_0 = 220 km/s, R_0 = 8.5 kpc):

    R = R_0 V_0 sin(l) / (V_0 sin(l) + v_LSR)
    d = R_0 cos(l) +/- sqrt(R^2 - R_0^2 sin^2(l))

For each (l, v_LSR) cell we deposit T_B at the implied (x, y).

Restricted to the outer Galaxy (R >= R_0) where the heliocentric
distance is single-valued; positions inside the solar circle are
ambiguous (near + far solution) and excluded.

In [ ]:
R0_KPC = 8.5
V0_KMS = 220.0
SIN_THRESH = 0.05   # |sin(l)| floor (kinematic-deprojection singularity guard)
HPBW_DEG = 3.4
HPBW_RAD = np.deg2rad(HPBW_DEG)

XY_RANGE_KPC = 18.0
XY_DXY_KPC = 0.10   # display grid (fine; effective resolution scales with d)

xy_edges = np.arange(-XY_RANGE_KPC, XY_RANGE_KPC + XY_DXY_KPC, XY_DXY_KPC)
xy_centers = 0.5 * (xy_edges[:-1] + xy_edges[1:])
nbin = len(xy_centers)

TB_sum = np.zeros((nbin, nbin), dtype=float)
TB_cnt = np.zeros((nbin, nbin), dtype=int)


def deproject_outer(l_deg, v_lsr):
    """Outer-Galaxy (R, d) under flat rotation. NaN elsewhere."""
    l = np.deg2rad(l_deg)
    sl = np.sin(l)
    cl = np.cos(l)
    bad = np.abs(sl) < SIN_THRESH
    denom = V0_KMS * sl + v_lsr
    with np.errstate(invalid='ignore', divide='ignore'):
        R = R0_KPC * V0_KMS * sl / denom
    outer = (R >= R0_KPC) & (v_lsr * sl <= 0)
    disc = R ** 2 - (R0_KPC * sl) ** 2
    with np.errstate(invalid='ignore'):
        d_plus = R0_KPC * cl + np.sqrt(np.maximum(disc, 0.0))
    keep = outer & (disc >= 0) & (d_plus > 0) & (~bad)
    return np.where(keep, R, np.nan), np.where(keep, d_plus, np.nan)


# Each (l, v) sample is splatted as a uniform top-hat disk of radius
# r(d) = d * HPBW_RAD / 2 (the beam half-width at distance d). Effective
# pixel size grows with distance instead of being fixed at XY_DXY_KPC.
l_unwrapped_fine = l_fine.astype(float) % 360.0
v_grid = v_lsr_overlap

n_kept = 0
R_min, R_max = np.inf, -np.inf
for i, l_i in enumerate(l_unwrapped_fine):
    col = lv_image_dense[:, i]
    finite = np.isfinite(col)
    if not finite.any():
        continue
    R_i, d_i = deproject_outer(np.full_like(v_grid, l_i), v_grid)
    valid = finite & np.isfinite(R_i) & np.isfinite(d_i)
    if not valid.any():
        continue
    l_rad = np.deg2rad(l_i)
    d_v = d_i[valid]
    R_v = R_i[valid]
    tb_v = col[valid]
    x_c = d_v * np.sin(l_rad)
    y_c = R0_KPC - d_v * np.cos(l_rad)
    r_pix = np.maximum(d_v * HPBW_RAD * 0.5, XY_DXY_KPC * 0.5)

    R_min = min(R_min, R_v.min())
    R_max = max(R_max, R_v.max())
    n_kept += int(d_v.size)

    for k in range(d_v.size):
        r = r_pix[k]
        ix_lo = max(0, int(np.floor((x_c[k] - r - xy_edges[0]) / XY_DXY_KPC)))
        ix_hi = min(nbin, int(np.ceil((x_c[k] + r - xy_edges[0]) / XY_DXY_KPC)))
        iy_lo = max(0, int(np.floor((y_c[k] - r - xy_edges[0]) / XY_DXY_KPC)))
        iy_hi = min(nbin, int(np.ceil((y_c[k] + r - xy_edges[0]) / XY_DXY_KPC)))
        if ix_hi <= ix_lo or iy_hi <= iy_lo:
            continue
        xs = xy_centers[ix_lo:ix_hi]
        ys = xy_centers[iy_lo:iy_hi]
        XX, YY = np.meshgrid(xs, ys, indexing='xy')
        inside = (XX - x_c[k]) ** 2 + (YY - y_c[k]) ** 2 <= r * r
        if not inside.any():
            continue
        TB_sum[iy_lo:iy_hi, ix_lo:ix_hi][inside] += tb_v[k]
        TB_cnt[iy_lo:iy_hi, ix_lo:ix_hi][inside] += 1

TB_face = np.where(TB_cnt > 0, TB_sum / np.maximum(TB_cnt, 1), np.nan)
n_filled = int((TB_cnt > 0).sum())
print(f'Deprojected {n_kept} (l, v) samples; beam-scaled splatting')
print(f'R range: [{R_min:.2f}, {R_max:.2f}] kpc   (R_0 = {R0_KPC} kpc)')
print(f'Filled {n_filled}/{nbin * nbin} pixels '
      f'({100 * n_filled / (nbin * nbin):.1f}%)')


def _wedge_poly(l0, l1, d_far):
    """Polygon vertices (x, y) for the wedge from Sun at longitudes l0..l1."""
    if l1 < l0:
        l1 += 360.0
    ls = np.linspace(l0, l1, max(2, int(np.ceil((l1 - l0) / 0.5)) + 1))
    ls_rad = np.deg2rad(ls)
    xs = d_far * np.sin(ls_rad)
    ys = R0_KPC - d_far * np.cos(ls_rad)
    poly_x = np.concatenate(([0.0], xs, [0.0]))
    poly_y = np.concatenate(([R0_KPC], ys, [R0_KPC]))
    return poly_x, poly_y


# --- Inaccessible-longitude wedges (Leuschner alt/az limits) ---
L_PROBE = np.arange(0, 360, 0.5)
inacc_probe = never_observable_mask(L_PROBE, 0.0)
runs_l = []
k = 0
while k < len(L_PROBE):
    if inacc_probe[k]:
        k0 = k
        while k < len(L_PROBE) and inacc_probe[k]:
            k += 1
        runs_l.append((L_PROBE[k0], L_PROBE[k - 1] + 0.5))
    else:
        k += 1

# --- Kinematic-singularity wedges (|sin(l)| < SIN_THRESH around l=0, l=180) ---
L_SING_HALF = np.rad2deg(np.arcsin(SIN_THRESH))
sing_runs = [
    ((-L_SING_HALF) % 360.0, L_SING_HALF),
    (180 - L_SING_HALF, 180 + L_SING_HALF),
]

D_FAR = 1.5 * (XY_RANGE_KPC + R0_KPC)
inacc_polys = [_wedge_poly(l0, l1, D_FAR) for l0, l1 in runs_l]
sing_polys  = [_wedge_poly(l0, l1, D_FAR) for l0, l1 in sing_runs]

# --- Plot ---
fig, ax = plt.subplots(figsize=(8, 8))
finite_face = TB_face[np.isfinite(TB_face)]
vmax = np.nanpercentile(finite_face, 99) if finite_face.size else 1.0

im = ax.imshow(
    TB_face, origin='lower',
    extent=[xy_edges[0], xy_edges[-1], xy_edges[0], xy_edges[-1]],
    cmap='inferno', vmin=0, vmax=vmax, interpolation='nearest',
)

first_inacc = True
for poly_x, poly_y in inacc_polys:
    label = 'Never observable' if first_inacc else None
    ax.fill(poly_x, poly_y, color='red', alpha=0.08, zorder=3)
    with mpl.rc_context({'hatch.color': 'red', 'hatch.linewidth': 0.5}):
        ax.fill(poly_x, poly_y, facecolor='none', edgecolor='none',
                hatch='///', alpha=0.4, zorder=4, label=label)
    first_inacc = False

first_sing = True
for poly_x, poly_y in sing_polys:
    label = (f'|sin(l)| < {SIN_THRESH} (singularity)'
             if first_sing else None)
    ax.fill(poly_x, poly_y, color='red', alpha=0.08, zorder=3)
    with mpl.rc_context({'hatch.color': 'red', 'hatch.linewidth': 0.5}):
        ax.fill(poly_x, poly_y, facecolor='none', edgecolor='none',
                hatch='\\\\\\', alpha=0.4, zorder=4, label=label)
    first_sing = False

theta = np.linspace(0, 2 * np.pi, 361)
ax.plot(R0_KPC * np.cos(theta), R0_KPC * np.sin(theta),
        ls='--', color='cyan', lw=0.8, alpha=0.7,
        label=f'Solar circle (R_0 = {R0_KPC} kpc)', zorder=5)
ax.plot(0, 0, marker='+', color='white', ms=10, mew=2,
        label='Galactic Center', zorder=6)
ax.plot(0, R0_KPC, marker='o', color='yellow', ms=8, mec='black', mew=0.7,
        label='Sun', zorder=6)

ax.set_xlabel('x [kpc]')
ax.set_ylabel('y [kpc]')
ax.set_title('Top-down (face-on) view of the Milky Way disk\n'
             f'beam-scaled splat (radius = d * HPBW/2, HPBW = {HPBW_DEG} deg)')
ax.set_aspect('equal')
ax.set_xlim(-XY_RANGE_KPC, XY_RANGE_KPC)
ax.set_ylim(-XY_RANGE_KPC, XY_RANGE_KPC)
ax.legend(loc='upper left', fontsize=9, framealpha=0.9)
cbar = fig.colorbar(im, ax=ax, pad=0.02, shrink=0.85)
cbar.set_label(r'$T_B$ [K]  (mean per (x, y) bin)')
plt.tight_layout()
plt.show()